# Registering Vector Data into eoAPI

This notebook demonstrates an end-to-end workflow for taking a vector dataset (like a zipped shapefile), ingesting it into a PostGIS database, and registering it as a SpatioTemporal Asset Catalog (STAC) Collection and Item within an eoAPI environment.

### Overview of Steps:
1. **Setup & Configuration:** Establish connections to the PostGIS database and STAC API.
2. **Data Acquisition:** Download a sample vector dataset.
3. **Database Ingestion:** Read the vector data with GeoPandas and push it to PostGIS (which powers the TiPg vector tile server).
4. **STAC Collection:** Create a metadata collection to group the vector data.
5. **STAC Item & Links:** Create the spatial metadata item, attaching the Mapbox Vector Tile links and Mapbox Style links (using the Render extension).
6. **Registration:** POST the item to the STAC API.
7. **Update Extents:** Automatically update the Collection's bounding box and temporal extent based on the ingested items.

## 1. Setup & Configuration
First, we import the necessary libraries. We are relying heavily on `geopandas` for spatial data manipulation, `sqlalchemy` for database connections, and `pystac` for building the metadata standard.

In [ ]:
import geopandas as gpd
from sqlalchemy import create_engine, inspect, text
from geoalchemy2 import Geometry
import pandas as pd
import pyarrow.parquet as pq
import time
import requests
from pystac import Collection, Extent, SpatialExtent, TemporalExtent, Link, Item
from datetime import datetime, timezone
from shapely.geometry import box, mapping
import os

We read environment variables to configure our connection to the `eoAPI` PostGIS database and define the target STAC API URLs and collection IDs.

In [ ]:
# Configuration
user = os.getenv("eoapi-db_user")
password = os.getenv("eoapi-db_password")
host = os.getenv("eoapi-db_host")
port = os.getenv("eoapi-db_port")
database = os.getenv("eoapi-db_dbname")

engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{database}")
inspector = inspect(engine)

STAC_API_URL = "http://eoapi-rw-stac:8080" 
VECTOR_SOURCE_URL = "https://data.statistik.gv.at/data/OGDEXT_NUTS_1_STATISTIK_AUSTRIA_NUTS3_20200101.zip"
FILE_PATH = f"{os.getenv('HOME')}/bucket/samples/OGDEXT_NUTS_1_STATISTIK_AUSTRIA_NUTS3_20200101.zip"
COLLECTION_ID = "demo_vector"
STYLE_URL = "https://raw.githubusercontent.com/eoxhub-workspaces/eoxhub-notebooks/refs/heads/main/assets/vector_style.json"


## 2. Download Sample Vector Data
Here we download a sample zipped shapefile (Austrian NUTS3 regions). If the file already exists in the mounted bucket, it skips the download to save time.

In [ ]:
# First we download the example vector file and put it into the mounted bucket
# You can also use the File Browser to upload your data
# (should be used for files smaller then 1 GB)
if os.path.exists(FILE_PATH):
    print(f"File exists at {FILE_PATH}. Skipping.")
else:
    print(f"Downloading to {FILE_PATH}...")
    with requests.get(VECTOR_SOURCE_URL, stream=True) as r:
        r.raise_for_status()
        with open(FILE_PATH, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"Download finished")

## 3. Process and Ingest Data into PostGIS
We load the downloaded shapefile using GeoPandas. Because TiPg (the vector tile engine in eoAPI) works best with lowercase column names and standardized projections, we clean the columns and reproject the geometries to EPSG:4326 (WGS84) before pushing it directly to the PostGIS database.

In [ ]:
# Load zipped shapefile in this case, most other vector formats should work
# like geojson, (geo)parquet, flatgebuf

gdf = gpd.read_file(FILE_PATH)
# in this step we make sure column names are set lower case as this can create issues with TiPg
gdf.columns = [c.lower() for c in gdf.columns]

# Ensure it is in WGS84 for STAC compliance
if gdf.crs and not gdf.crs.is_geographic:
    gdf = gdf.to_crs(epsg=4326)

# Insert into TiPg (eoAPI) database using GeoPandas to_postgis 
gdf.to_postgis(
    name=COLLECTION_ID,
    con=engine,
    if_exists="replace",
    index=False,
    dtype={"geometry": Geometry("GEOMETRY", srid=4326)}
)

In [ ]:
# We can have a quick look at the created db table
columns = inspector.get_columns(COLLECTION_ID)
for column in columns:
    print(f"Column name: {column['name']}, Type: {column['type']}")

## 4. Create a STAC Collection
Before we can register the individual vector item, we need a Collection to hold it. We check if `demo_vector` already exists. If not, we define its metadata (like spatial and temporal extents) and create it via a POST request to the STAC API.

In [ ]:
# Now we create the collection that will point to the vector tiles (only if it does not already exist)
check_url = f"{STAC_API_URL}/collections/{COLLECTION_ID}"
resp = requests.get(check_url)
if resp.status_code == 200:
    print(f"Collection '{COLLECTION_ID}' exists.")
else:
    print(f"Creating collection '{COLLECTION_ID}'...")
    coll = Collection(
        id=COLLECTION_ID,
        title="Demo Vector Collection",
        description="Demo to show loading of vector data into eoAPI",
        extent=Extent(SpatialExtent([[-180, -90, 180, 90]]), TemporalExtent([[datetime(2020,1,1,tzinfo=timezone.utc), None]])),
        license="CC-BY-4.0"
    )
    # Sending a POST request with the Collection json to create the collection
    resp = requests.post(f"{STAC_API_URL}/collections", json=coll.to_dict())

    if resp.ok: # You can also use: if resp.status_code in [200, 201]:
        print(f"Success! Collection '{COLLECTION_ID}' created.")
    else:
        print(f"Failed with status {resp.status_code}.")
        print(f"Error details: {resp.text}")

In [ ]:
# If we wanted to delete the collection we could use following command
# requests.delete(f"{STAC_API_URL}/collections/{COLLECTION_ID}")

## 5. Generate the STAC Item
We create a STAC Item to represent our dataset. Because this is vector data served as tiles, we don't need to embed the full complex geometry in the STAC Item; the bounding box (bbox) is sufficient.

We also add two critical `pystac.Link` objects here:
* A **Vector Tile Link** (`application/vnd.mapbox-vector-tile`) pointing to our TiPg endpoint.
* A **Style Link** (`text/vector-styles`) that provides a JSON style spec so frontends know how to color and render the vectors.

In [ ]:
# We need a STAC item definition in order to register it into the collection
# see https://github.com/radiantearth/stac-spec/blob/master/item-spec/item-spec.md for further details

# Extract Bounding Box (West, South, East, North)
bbox = list(gdf.total_bounds) # [minx, miny, maxx, maxy]
geom_poly = box(*bbox)
# Create geometry based on bbox
geometry_dict = mapping(geom_poly)

# Create the STAC Item
item = Item(
    id="vector_tiles_item",
    geometry=geometry_dict,
    bbox=bbox,
    datetime=datetime.utcnow(), # Or extract from file attributes if available
    properties={},
    collection=COLLECTION_ID
)

# Add preview link to allow rendering data in eoAPI GUI
eoapi_vector_endpoint = os.getenv("RASTER_ENDPOINT", "").replace("raster", "vector")

vector_tiles = (
    f"{eoapi_vector_endpoint}/collections/public.{COLLECTION_ID}/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}"
)

vt_link = Link(
    rel="vector-tile",
    target=vector_tiles,
    title=COLLECTION_ID,
    media_type="application/vnd.mapbox-vector-tile",
    extra_fields={
        "key": COLLECTION_ID  # This moves the 'key' into the JSON properties
    }
)
item.add_link(vt_link)

style_link = Link(
    rel="style",
    target=STYLE_URL,
    media_type="text/vector-styles",
    extra_fields={
        "links:keys": [COLLECTION_ID]
    }
)
item.add_link(style_link)


In [ ]:
# Helper to print item to see if everything is good to go
# item.to_dict()

## 6. Publish the STAC Item
With our STAC Item fully assembled and validated, we serialize it to a dictionary and POST it to the `eoAPI` STAC `/items` endpoint.

In [ ]:
# Now that the item was created we can pass it to eoAPI for ingestion

url = f"{STAC_API_URL}/collections/{COLLECTION_ID}/items"
response = requests.post(url, json=item.to_dict())

if response.status_code in [200, 201]:
    print("Successfully posted STAC Item.")
else:
    print(f"Error {response.status_code}: {response.text}")

## 7. Update Collection Extents
Finally, we run a utility function to fetch the items back from the collection, calculate their total spatial and temporal coverage, and patch the parent Collection. This ensures that web catalogs display the correct overview bounds.

In [ ]:
# Update temporal and spatial extent based on ingested datasets
import requests

def update_stac(url, cid):
    # 1. Fetch all items (limited to 1000 for brevity)
    items = requests.get(f"{url}/collections/{cid}/items?limit=1000").json()['features']
    
    # 2. Extract and calculate (flatten coordinates and collect dates)
    coords = [pt for f in items for pt in f['geometry']['coordinates'][0]]
    dts = [f['properties']['datetime'] for f in items]
    
    # 3. Patch the collection
    ext = {
        "spatial": {"bbox": [[min(c[0] for c in coords), min(c[1] for c in coords), 
                             max(c[0] for c in coords), max(c[1] for c in coords)]]},
        "temporal": {"interval": [[min(dts), max(dts)]]}
    }
    
    r = requests.patch(f"{url}/collections/{cid}", json={"extent": ext})
    print("Updated" if r.ok else f"Error: {r.text}")

update_stac(STAC_API_URL, COLLECTION_ID)